# RK1828 Qwen3-Reranker Demo


[Qwen3-Reranker-0.6B](https://github.com/QwenLM/Qwen3-Embedding): scores a
(query, document) pair with a 0~1 relevance score for RAG reranking. GRQ W4A16
quantized, measured 132 ms prefill per pair; relevant 0.9985 vs irrelevant 0.0000.

| Step | Content |
|---|---|
| 1 | Download models (from huggingface.co) |
| 2 | Extract the prebuilt demo |
| 3 | Enter a query and candidate documents |
| 4 | Score and rank |

## 0. Environment check

Make sure the RKNN3 runtime (`librknn3_api.so`) and the RK1828 device are ready.
If this fails, install the driver/runtime first (the `rknn3-rk182x` DEB package)
and check with `sudo rknn-smi info` that the RK1828 shows up.

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # loads /usr/lib/librknn3_api.so; failure means the runtime is not installed

devices = find_devices()
if not devices:
    raise RuntimeError("No RK1828 device found! Check: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"Found device: id={dev_id}  interface={dev_type}")

## 1. Download models

The models for this demo are hosted on Hugging Face (account `qhddz`) and are
downloaded directly from **huggingface.co**. Run the next cell to start.

> Files that already exist (and are non-empty) are **skipped automatically** —
> re-running never downloads them twice.
> If your network cannot reach huggingface.co, set up a proxy first, e.g.
> `export https_proxy=http://<your-proxy>:<port>` before starting Jupyter.

In [ ]:
# (HF repo, path in repo, local path)
FILES = [
    ("qwen3-reranker-0.6b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_reranker/demo_x86_64.tar.gz"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.embed.bin", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.embed.bin"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.rknn", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.rknn"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.tokenizer.gguf", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.tokenizer.gguf"),
    ("qwen3-reranker-0.6b-rknn3-rk1828", "model/Qwen3-Reranker-0.6B.weight", "demos/qwen3_reranker/model/Qwen3-Reranker-0.6B.weight"),
]

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, "hf")

## 2. Extract the prebuilt demo

`demo_x86_64.tar.gz` contains the official x86_64 C++ demo and its runtime
libraries (`lib/`). Hugging Face does not host raw executables, so they are
shipped as a tarball; extract once (re-running skips automatically).

> **Why a C++ demo instead of pure Python?**
> Notebooks 01-05 (classification / detection / FastSAM / pose / segmentation)
> are a single model with one forward pass and simple post-processing, so they
> walk through everything in Python (`rknn3.py`). This demo's pipeline is far
> more complex — chained models, tokenizer, KV cache, sliding-window /
> streaming state, hundreds of lines of orchestration — reimplementing it in
> Python would be costly and risk diverging from the official behavior.
> So we run the official `rknn3-model-zoo` C++ demo directly; source code:
> [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> under `examples/`.

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_reranker/demo_x86_64.tar.gz", "demos/qwen3_reranker")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. Enter a query and candidate documents (one per line)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_query = widgets.Text(value="What are gravitational waves?",
                       description="Query:", layout=widgets.Layout(width="75%"))
w_docs = widgets.Textarea(
    value="Gravitational waves are ripples in spacetime caused by accelerating masses.\n"
          "The best chocolate chip cookie recipe uses brown butter.\n"
          "LIGO first detected gravitational waves in 2015, confirming Einstein's prediction.\n"
          "Python is a popular programming language for data science.",
    description="Docs (one per line):", layout=widgets.Layout(width="75%", height="120px"))
display(w_query, w_docs)
print("Then keep running downwards ▼")

## 4. Score and rank

In [ ]:
import re

import numpy as np

DEMO = "demos/qwen3_reranker"
docs = [d.strip() for d in w_docs.value.splitlines() if d.strip()]
scores = []
for d in docs:
    p = run_demo(["./rknn_qwen3_reranker_demo",
                  "model/Qwen3-Reranker-0.6B.rknn", "model/Qwen3-Reranker-0.6B.weight",
                  "model/Qwen3-Reranker-0.6B.tokenizer.gguf",
                  "model/Qwen3-Reranker-0.6B.embed.bin",
                  "0xff", w_query.value, d], DEMO)
    nums = re.findall(r"(?:score|Score|relevance)[:\s]+([0-9]*\.?[0-9]+)", p.stdout)
    if not nums:  # fallback: take the last 0~1 decimal in the output
        nums = re.findall(r"(?<![\d.])(0?\.[0-9]+|1\.0+)(?![\d.])", p.stdout)
    scores.append(float(nums[-1]))
    print(f"  -> {scores[-1]:.4f}")

order = np.argsort(scores)[::-1]
print("\nRanked by relevance:")
for i in order:
    print(f"  {scores[i]:.4f}  {docs[i][:60]}")

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
labels = [d[:28] for d in docs]
order = np.argsort(scores)
plt.figure(figsize=(8, 3.5))
plt.barh([labels[i] for i in order], [scores[i] for i in order], color="#3b82f6")
plt.xlim(0, 1)
plt.title(f"query: {w_query.value}")
plt.tight_layout()
plt.show()

## Summary

- A reranker is the "fine ranking" stage of RAG: embeddings shortlist top-K,
  the reranker re-scores them one by one for precision.
- The score gap between relevant and irrelevant is extreme (0.99 vs 0.00),
  showing excellent discrimination.